# SAP pipeline validation — Monte Carlo against known ground truth

**Purpose.** Prove — not assert — that the estimators in `rct_data_literacy_analysis_3.ipynb`
hold their nominal Type-I error, recover known truth, and achieve the power the thesis already
claims (`power_analysis_v2.py` / `power_results_v2.txt` / `app:power`). The analysis notebook has
only been shown to *run*; this notebook shows the code *behaves*.

**Design rule: no re-implementation.** Every fit below calls `sap_estimators.py` — the same
module `rct_data_literacy_analysis_3.ipynb` imports. The validation therefore exercises the
exact code that will run on real data. (The two v3-inline pieces — the VB corroboration model
and the Wald CACE — are copied verbatim where tested and marked as such.)

**Guardrails honoured.**
- *Simulation only*: this notebook never touches the participant CSV.
- *DGPs mirror the SAP equations*: the accuracy DGP **is** `eq:primary-acc` (normal LMM with
  participant intercept, wave dummy, two centred covariates); the success-time DGP **is** a
  proportional-hazards competing-risks process matching `eq:primary-st`'s cause-specific
  estimand; the Bloom DGP replicates `power_analysis_v2.py`'s own H4a simulation (which has
  **no item random effect** — a stated deviation from `eq:bloom`'s $v_m$, inherited from the
  power script so its numbers are reproducible).
- *Discrepancies are reported, never patched into the thesis text*: see the final report.

**Known reconciliation issue found while reading the power script (adjudicated in Check 3):**
`power_analysis_v2.py` line 139 computes NI power with `alpha=0.025` (a 95%-CI decision rule),
while its own caption — and the SAP, and v3 — specify the **90% CI rule (one-sided α = 0.05)**.
The pre-registered 0.792 therefore belongs to a *stricter* test than the one that will be run.
Check 3 estimates the empirical rate under **both** rules in the same replicates.

**Monte Carlo budget.** Replicate counts per check are chosen so the MC standard error of every
quoted rate is conclusive (rate MCSE = √(p(1−p)/R)): R=1200 → ±0.006 at p=.05; R=600–800 →
±0.008–0.009 at p=.05 / ±0.018–0.020 at p=.5; coverage at R=400–500 → ±0.010–0.011. The
expensive engines run at reduced R with the correspondingly wider MCSE reported, not hidden:
VB GLMM R=100 (calibration-ratio MCSE ≈ 0.07 — conclusive against the ~0.5–0.7 anticonservatism
at issue), MICE R=300, double-fit Cox R=400. A smoke-tested single fit costs ≈0.25 s (LMM),
≈0.15 s (GEE), ≈1.4 s (clustered Cox, two waves), ≈10 s (VB); total budgeted wall-clock ≈ 60–80
minutes. Every check reports its failure (non-convergence) count.

## 0. Setup

In [1]:
import time, warnings
import numpy as np
import pandas as pd
import scipy.stats as st
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore")

import sap_estimators as sap   # THE module under test (shared with the analysis notebook)

SEED   = 20260711
SD_ACC = 0.2089          # D8 beta-binomial derivation (power_results_v2.txt line 4)
P0     = 0.60            # assumed control accuracy
R2     = 0.30            # covariate R^2 (Mini-VLAT + AIUse)
RHO    = 0.50            # within-person wave-wave correlation (power script rho)
N_ARM  = 66              # pre-registered design
ALPHA  = 0.05
DELTA  = 0.085

REPS = dict(C1=1200, C1B=600, C2A=800, C2B=300, BLOOM=600, C3=600, C3IV=500,
            C4=600, C4C=500, C5A=500, C5B=400, C5C=500, C6VB=100, C7MAR=300, C7MNAR=600)
import os
if os.environ.get("SAP_VAL_FAST"):        # smoke-test mode: tiny reps, same code paths
    REPS = {k: max(8, v // 100) for k, v in REPS.items()}
    print("*** SAP_VAL_FAST smoke-test mode: REPS =", REPS, "— results NOT interpretable ***")

RESULTS, DISCREPANCIES, TIMES = [], [], {}

def rate_mcse(p, n): return float(np.sqrt(max(p*(1-p), 1e-12)/n))

def record(check, dgp, nominal, empirical, mcse, verdict, note=""):
    RESULTS.append(dict(check=check, dgp=dgp, nominal=nominal,
                        empirical=empirical, mc_se=mcse, verdict=verdict, note=note))
    print(f"  [{check}] {dgp}: nominal={nominal}  empirical={empirical}  MCSE={mcse}  -> {verdict}"
          + (f"  ({note})" if note else ""))

def v_size(emp, nominal, mcse):
    """Size check: anticonservative = fail; conservative noted."""
    if emp <= nominal + 2*mcse:
        return "PASS" + (" (conservative)" if emp < nominal - 2*mcse else "")
    return "FAIL (anticonservative)"

def v_target(emp, nominal, mcse, tol=0.03):
    return "PASS (within MC error)" if abs(emp - nominal) <= max(2*mcse, tol) else "FLAG (diverges)"

def v_cover(emp, mcse):
    return "PASS" if 0.95 - max(2*mcse, .02) <= emp <= 0.95 + max(2*mcse, .02) else "FAIL"

print("module under test:", sap.__file__)
print("statsmodels", sm.version.version if hasattr(sm, "version") else "", "| replicates:", REPS)

module under test: /private/tmp/claude-501/-Users-brunokneffel-Library-Mobile-Documents-com-apple-CloudDocs-gymnasium-steglitz-B-SC-Frankfurt-School-Oxford-Thesis-WIP-thesis-latex/44677145-89cf-4806-9120-e42c6ec17263/scratchpad/sap_estimators.py
statsmodels  | replicates: {'C1': 1200, 'C1B': 600, 'C2A': 800, 'C2B': 300, 'BLOOM': 600, 'C3': 600, 'C3IV': 500, 'C4': 600, 'C4C': 500, 'C5A': 500, 'C5B': 400, 'C5C': 500, 'C6VB': 100, 'C7MAR': 300, 'C7MNAR': 600}


## 0.1 Shared data-generating processes

**Accuracy DGP = `eq:primary-acc` verbatim.**
$Y_{iw} = 0.60 + \beta_1 S_i + \beta_2 U_i + \delta W_w + \beta_4 S_iW_w + \beta_5 U_iW_w
+ \gamma_1 C_{1i} + \gamma_2 C_{2i} + u_i + \varepsilon_{iw}$, with variance components
calibrated to the power script: total SD = 0.2089; the two covariates jointly explain
$R^2=0.30$ ($\gamma_j = SD\sqrt{0.15}$, $C_j\sim N(0,1)$); $\sigma_u^2=(\rho-R^2)\,SD^2$ and
$\sigma_\varepsilon^2=(1-\rho)\,SD^2$ give wave–wave correlation $\rho=0.5$. A binomial variant
draws $16\cdot Y$ from Binomial(16, clip(μ)) to test size under the real outcome's granularity.

**Success-time DGP = cause-specific PH process (`eq:primary-st`'s estimand).** Per item,
correct-answer time $T_c\sim\mathrm{Exp}(0.055\,e^{\theta_{arm,wave}+b_i})$ competes with
wrong-submission time $T_w\sim\mathrm{Exp}(0.045)$; the earlier one is observed (event=1 iff
correct), admin-censored at 120 s. Frailty $b_i\sim N(0,\sigma_b^2)$ sits on the correct-cause
hazard only (ability: better participants are both more often and faster correct), which
generates within-participant clustering in *both* event status and durations.

**Bloom DGP = `power_analysis_v2.py`'s own H4a simulation, plus a Socratic arm at control
levels.** Cell probabilities Google (.80 low / .50 high), Unrestricted (.75 low /
.50 − .05 − extra high); person effect $\theta_i\sim N(0,0.7^2)$ on the logit; 8+8 Bernoulli
items. No item random effect (deviation from `eq:bloom`, inherited from the power script).

In [2]:
ARMS = np.repeat(["google", "socratic", "unrestricted"], N_ARM)
SOC  = (ARMS == "socratic").astype(float)
UNR  = (ARMS == "unrestricted").astype(float)
NTOT = 3 * N_ARM
G_LOAD = SD_ACC * np.sqrt(R2 / 2)             # per-covariate loading
SIG_U  = np.sqrt((RHO - R2) * SD_ACC**2)      # person intercept SD
SIG_E  = np.sqrt((1 - RHO) * SD_ACC**2)       # within-wave residual SD

def sim_acc_trial(rng, b_soc=0.0, b_unr=0.0, d_wave=0.03, b_soc_w=0.0, b_unr_w=0.0,
                  binomial=False, drop_delayed=None):
    """One trial as an acc_pw frame. b_* may be scalars or per-participant arrays
    (length N_ARM, applied to that arm's block). drop_delayed: None | float rate |
    callable(y_imm, y_del, u, eps_del, c1) -> boolean keep-mask for the delayed wave."""
    c1, c2 = rng.normal(0, 1, NTOT), rng.normal(0, 1, NTOT)
    u = rng.normal(0, SIG_U, NTOT)
    bsoc = np.zeros(NTOT); bsoc[N_ARM:2*N_ARM] = b_soc
    bunr = np.zeros(NTOT); bunr[2*N_ARM:]      = b_unr
    bsw  = np.zeros(NTOT); bsw[N_ARM:2*N_ARM]  = b_soc_w
    buw  = np.zeros(NTOT); buw[2*N_ARM:]       = b_unr_w
    eps_i, eps_d = rng.normal(0, SIG_E, NTOT), rng.normal(0, SIG_E, NTOT)
    mu_i = P0 + bsoc + bunr + G_LOAD*c1 + G_LOAD*c2 + u + eps_i
    mu_d = P0 + bsoc + bunr + d_wave + bsw + buw + G_LOAD*c1 + G_LOAD*c2 + u + eps_d
    if binomial:
        y_i = rng.binomial(16, np.clip(mu_i, 0.02, 0.98)) / 16.0
        y_d = rng.binomial(16, np.clip(mu_d, 0.02, 0.98)) / 16.0
    else:
        y_i, y_d = mu_i, mu_d
    keep_d = np.ones(NTOT, bool)
    if callable(drop_delayed):
        keep_d = drop_delayed(y_i, y_d, u, eps_d, c1, rng)
    elif drop_delayed:
        keep_d = rng.random(NTOT) >= drop_delayed
    base = dict(pid=np.arange(NTOT), arm=ARMS, socratic=SOC, unrestricted=UNR,
                minivlat_c=c1, aiuse_c=c2)
    fi = pd.DataFrame({**base, "wave_d": 0, "acc": y_i})
    fd = pd.DataFrame({**base, "wave_d": 1, "acc": y_d})[keep_d]
    return pd.concat([fi, fd], ignore_index=True)

def sim_cox_trial(rng, n_arm=N_ARM, th_unr=(np.log(0.75), np.log(0.90)),
                  th_soc=(0.0, 0.0), sigma_b=0.0, lam_c=0.055, lam_w=0.045,
                  cens=120.0, n_items=16, waves=(0, 1), arms=3):
    """Item-level competing-risks frame matching eq:primary-st's columns.
    th_* = (immediate log-HR, delayed log-HR) on the correct-cause hazard."""
    n = arms * n_arm
    arm = np.repeat(["google", "socratic", "unrestricted"][:arms], n_arm)
    soc = (arm == "socratic").astype(float); unr = (arm == "unrestricted").astype(float)
    b = rng.normal(0, sigma_b, n) if sigma_b > 0 else np.zeros(n)
    c1, c2 = rng.normal(0, 1, n), rng.normal(0, 1, n)
    rows = []
    for wd in waves:
        th = soc * (th_soc[wd]) + unr * (th_unr[wd])
        for m in range(n_items):
            tc = rng.exponential(1.0 / (lam_c * np.exp(th + b)))
            tw = rng.exponential(1.0 / lam_w, n)
            event = (tc <= np.minimum(tw, cens)).astype(int)
            dur = np.where(event == 1, tc, np.minimum(tw, cens))
            rows.append(pd.DataFrame(dict(
                pid=np.arange(n), arm=arm, socratic=soc, unrestricted=unr,
                wave=("immediate" if wd == 0 else "delayed"), wave_d=wd,
                soc_wave=soc * wd, unr_wave=unr * wd,
                minivlat_c=c1, aiuse_c=c2, duration_s=dur, event=event)))
    return pd.concat(rows, ignore_index=True)

BLOOM_CELLS = dict(google=(0.80, 0.50), socratic=(0.80, 0.50))  # + unrestricted per 'extra'
def sim_bloom_trial(rng, extra_pp, n_arm=N_ARM, sigma_p=0.7):
    """power_analysis_v2.py H4a DGP + Socratic at control levels; immediate wave only."""
    from scipy.special import expit, logit
    pU = (0.75, 0.50 - 0.05 - extra_pp / 100.0)
    cells = {**BLOOM_CELLS, "unrestricted": pU}
    frames = []
    for a_i, a in enumerate(["google", "socratic", "unrestricted"]):
        th = rng.normal(0, sigma_p, n_arm)
        pl = expit(logit(cells[a][0]) + th)
        ph = expit(logit(cells[a][1]) + th)
        low  = rng.binomial(1, np.repeat(pl, 8)).astype(int)
        high = rng.binomial(1, np.repeat(ph, 8)).astype(int)
        pid = a_i * n_arm + np.arange(n_arm)
        c1, c2 = rng.normal(0, 1, n_arm), rng.normal(0, 1, n_arm)
        for vals, hi in [(low, 0), (high, 1)]:
            frames.append(pd.DataFrame(dict(
                pid=np.repeat(pid, 8), arm=a,
                socratic=float(a == "socratic"), unrestricted=float(a == "unrestricted"),
                order_hi=hi, item_id=[f"{'H' if hi else 'L'}{j%8}" for j in range(8*n_arm)],
                correct=vals, minivlat_c=np.repeat(c1, 8), aiuse_c=np.repeat(c2, 8))))
    return pd.concat(frames, ignore_index=True)

print("simulators defined; variance components:",
      dict(g=round(G_LOAD,4), sig_u=round(SIG_U,4), sig_e=round(SIG_E,4)))

simulators defined; variance components: {'g': 0.0809, 'sig_u': 0.0934, 'sig_e': 0.1477}


## Check 1 — Type-I error of the gatekeeper (global null)
Global null: no arm, interaction, or covariate-confounded effects (a benign wave shift of
+3 pp is retained — orthogonal to all contrasts). R = 2000 (MCSE at .05 = ±.005). Full
compliance, so the per-protocol refit inside `gatekeeper()` runs on the identical sample —
the real code path. Under this null, a *false confirmatory claim* is: rejecting H1ₐ, or
(having falsely rejected) claiming the supporting contrast. The NI claim is **true** here
(Socratic−Google = 0 > −Δ), so its declaration rate is reported as gate-throughput, not error.
Also reported: bias and SE-calibration of β̂₂, its 95% CI coverage, and the H5ₐ one-sided size.

In [3]:
t0 = time.time(); rng = np.random.default_rng(SEED + 1)
R = REPS["C1"]; fail = 0
rej1 = ni = sup = h5 = cover = 0; b2s, se2s = [], []
for r in range(R):
    acc_pw = sim_acc_trial(rng)
    try:
        gk = sap.gatekeeper(acc_pw, set(acc_pw.pid), alpha=ALPHA, delta=DELTA)
        _, _, p5 = sap.h5a_simple_effect(gk["res"])
    except Exception:
        fail += 1; continue
    rej1 += gk["rej_h1a"]; ni += gk["ni_declared"]; sup += gk["rej_sup"]; h5 += (p5 < ALPHA)
    b2s.append(gk["b2"]); se2s.append(gk["se2"])
    lo, hi = gk["ci95_b2"]; cover += (lo <= 0.0 <= hi)
n = R - fail
p1, pni, psup, ph5, pcov = rej1/n, ni/n, sup/n, h5/n, cover/n
print(f"reps={n} (failures={fail})")
record("1 Type-I", "global null: H1a one-sided size", 0.05, round(p1,4), round(rate_mcse(p1,n),4), v_size(p1,0.05,rate_mcse(p1,n)))
fwer = p1  # false claims possible: H1a and (nested) supporting => FWER = P(reject H1a)
record("1 Type-I", "global null: FWER any false claim", "<=0.05", round(fwer,4), round(rate_mcse(fwer,n),4), v_size(fwer,0.05,rate_mcse(fwer,n)))
record("1 Type-I", "global null: supporting-contrast false claim", "<<0.05", round(psup,4), round(rate_mcse(max(psup,1e-4),n),4), v_size(psup,0.05,rate_mcse(max(psup,1e-4),n)), note="requires H1a+NI gates first")
record("1 Type-I", "global null: NI declared (TRUE claim; gate-throughput)", "<=P(rej H1a)", round(pni,4), round(rate_mcse(max(pni,1e-4),n),4), "PASS" if pni <= p1 + 1e-9 else "FAIL (gate leak)")
record("1 Type-I", "global null: H5a one-sided size", 0.05, round(ph5,4), round(rate_mcse(ph5,n),4), v_size(ph5,0.05,rate_mcse(ph5,n)))
bias = float(np.mean(b2s)); calib = float(np.mean(se2s)/np.std(b2s, ddof=1))
record("1 Type-I", "beta2_hat bias (proportion scale)", 0.0, round(bias,5), round(np.std(b2s,ddof=1)/np.sqrt(n),5), "PASS" if abs(bias) < 3*np.std(b2s,ddof=1)/np.sqrt(n) else "FAIL")
record("1 Type-I", "beta2 SE calibration (mean SE / SD est)", 1.0, round(calib,3), round(1/np.sqrt(2*(n-1)),3), "PASS" if abs(calib-1) < 0.05 else "FLAG")
record("1 Type-I", "beta2 95% CI coverage", 0.95, round(pcov,4), round(rate_mcse(pcov,n),4), v_cover(pcov, rate_mcse(pcov,n)))
TIMES["C1"] = time.time()-t0; print(f"[C1 done in {TIMES['C1']:.0f}s]")

reps=1200 (failures=0)
  [1 Type-I] global null: H1a one-sided size: nominal=0.05  empirical=0.0408  MCSE=0.0057  -> PASS
  [1 Type-I] global null: FWER any false claim: nominal=<=0.05  empirical=0.0408  MCSE=0.0057  -> PASS
  [1 Type-I] global null: supporting-contrast false claim: nominal=<<0.05  empirical=0.0075  MCSE=0.0025  -> PASS (conservative)  (requires H1a+NI gates first)
  [1 Type-I] global null: NI declared (TRUE claim; gate-throughput): nominal=<=P(rej H1a)  empirical=0.0217  MCSE=0.0042  -> PASS
  [1 Type-I] global null: H5a one-sided size: nominal=0.05  empirical=0.0608  MCSE=0.0069  -> PASS
  [1 Type-I] beta2_hat bias (proportion scale): nominal=0.0  empirical=0.0007  MCSE=0.00086  -> PASS
  [1 Type-I] beta2 SE calibration (mean SE / SD est): nominal=1.0  empirical=1.025  MCSE=0.02  -> PASS
  [1 Type-I] beta2 95% CI coverage: nominal=0.95  empirical=0.9558  MCSE=0.0059  -> PASS
[C1 done in 426s]


In [4]:
# Check 1b — size under the real outcome's granularity (16-item binomial proportions)
t0 = time.time(); rng = np.random.default_rng(SEED + 11)
R = REPS["C1B"]; fail = 0; rej1 = 0; sds = []
for r in range(R):
    acc_pw = sim_acc_trial(rng, binomial=True)
    sds.append(acc_pw.loc[acc_pw.wave_d == 0, "acc"].std())
    try:
        gk = sap.gatekeeper(acc_pw, set(acc_pw.pid), alpha=ALPHA, delta=DELTA)
    except Exception:
        fail += 1; continue
    rej1 += gk["rej_h1a"]
n = R - fail; p1 = rej1/n
print(f"reps={n} (failures={fail}); achieved immediate-wave SD(acc) = {np.mean(sds):.4f} "
      f"(binomial layer adds item noise on top of the 0.2089 latent scale)")
record("1 Type-I", "binomial-granularity null: H1a size", 0.05, round(p1,4),
       round(rate_mcse(p1,n),4), v_size(p1,0.05,rate_mcse(p1,n)),
       note="16-item Binomial outcome; LMM on proportions")
record("1 Type-I", "binomial-granularity: fit-failure rate (post optimizer-chain fix)", "~0",
       round(fail/R,4), round(rate_mcse(max(fail/R,1e-4),R),4), "PASS" if fail/R < 0.01 else "FAIL")
DISCREPANCIES.append(
    "PIPELINE DEFECT FOUND AND FIXED DURING VALIDATION: the original fit_primary_acc used a bare "
    "MixedLM.fit(method='lbfgs'); on realistic 16-item binomial-proportion outcomes this raised "
    "LinAlgError('Singular matrix') whenever the profiled random-intercept variance neared zero "
    "(100% of smoke-test draws). sap_estimators.py now carries a pre-specified optimizer fallback "
    "chain (lbfgs -> bfgs -> powell -> cg), identical wherever lbfgs succeeds. ACTION: add one line "
    "to the SAP software paragraph naming this chain before registration.")
TIMES["C1B"] = time.time()-t0; print(f"[C1b done in {TIMES['C1B']:.0f}s]")

reps=600 (failures=0); achieved immediate-wave SD(acc) = 0.2307 (binomial layer adds item noise on top of the 0.2089 latent scale)
  [1 Type-I] binomial-granularity null: H1a size: nominal=0.05  empirical=0.0633  MCSE=0.0099  -> PASS  (16-item Binomial outcome; LMM on proportions)
  [1 Type-I] binomial-granularity: fit-failure rate (post optimizer-chain fix): nominal=~0  empirical=0.0  MCSE=0.0004  -> PASS
[C1b done in 212s]


## Check 2 — Power recovery at the pre-registered effect sizes
**H1ₐ:** deficits of 7.6 pp (script MDES, target 0.80) and 17 pp (shen2026 anchor, target
≈ 1.00), R = 1000 / 400. The script's power is an immediate-wave ANCOVA-*t*; the pipeline's
is the LMM's immediate-anchored contrast — the per-rep immediate-wave ANCOVA is run alongside
so any divergence is attributable, not silent.
**H4ₐ Bloom:** the script's exact DGP at extra ∈ {0, 5, 10, 15} pp, R = 600 each. Each
replicate is analysed twice: the script's own test (Welch *t* on within-person high−low
difference scores) — which must reproduce 0.267 / 0.695 / 0.956 — and the pipeline's
confirmatory GEE (`fit_bloom_gee` + `h4a_from_gee`). A material t-vs-GEE gap is a **flag**:
the pre-registered H4ₐ power figures were computed under a different test than the SAP runs.

In [5]:
t0 = time.time(); rng = np.random.default_rng(SEED + 2)
for tag, eff, target, R in [("7.6pp (MDES)", -0.076, 0.80, REPS["C2A"]),
                            ("17pp (anchor)", -0.17, 0.999, REPS["C2B"])]:
    fail = 0; rej_lmm = rej_ols = 0
    for r in range(R):
        acc_pw = sim_acc_trial(rng, b_unr=eff, b_unr_w=0.0)
        try:
            gk = sap.gatekeeper(acc_pw, set(acc_pw.pid), alpha=ALPHA, delta=DELTA)
        except Exception:
            fail += 1; continue
        rej_lmm += gk["rej_h1a"]
        imm = acc_pw[acc_pw.wave_d == 0]
        o = smf.ols("acc ~ socratic + unrestricted + minivlat_c + aiuse_c", imm).fit()
        rej_ols += (st.norm.cdf(o.params["unrestricted"] / o.bse["unrestricted"]) < ALPHA)
    n = R - fail; pl, po = rej_lmm/n, rej_ols/n
    record("2 Power", f"H1a {tag}: pipeline LMM", target, round(pl,4), round(rate_mcse(pl,n),4), v_target(pl,target,rate_mcse(pl,n)))
    record("2 Power", f"H1a {tag}: script-style ANCOVA (same reps)", target, round(po,4), round(rate_mcse(po,n),4), v_target(po,target,rate_mcse(po,n)), note="attribution baseline")
TIMES["C2ab"] = time.time()-t0; print(f"[C2 H1a done in {TIMES['C2ab']:.0f}s]")

  [2 Power] H1a 7.6pp (MDES): pipeline LMM: nominal=0.8  empirical=0.7987  MCSE=0.0142  -> PASS (within MC error)
  [2 Power] H1a 7.6pp (MDES): script-style ANCOVA (same reps): nominal=0.8  empirical=0.7925  MCSE=0.0143  -> PASS (within MC error)  (attribution baseline)


  [2 Power] H1a 17pp (anchor): pipeline LMM: nominal=0.999  empirical=1.0  MCSE=0.0  -> PASS (within MC error)
  [2 Power] H1a 17pp (anchor): script-style ANCOVA (same reps): nominal=0.999  empirical=1.0  MCSE=0.0  -> PASS (within MC error)  (attribution baseline)
[C2 H1a done in 416s]


In [6]:
t0 = time.time(); rng = np.random.default_rng(SEED + 3)
BLOOM_KEEP = {}   # store extra=10 GEE (est,se) pairs for Check 6 reuse
targets_t = {0: 0.05, 5: 0.267, 10: 0.695, 15: 0.956}
for extra, tgt in targets_t.items():
    R = REPS["BLOOM"]; fail = 0; rej_t = rej_gee = 0; keep = []
    for r in range(R):
        ub = sim_bloom_trial(rng, extra)
        # (i) the power script's own analysis: Welch t on high-minus-low difference scores
        dsc = (ub[ub.order_hi==1].groupby(["pid","arm"], observed=True)["correct"].mean()
               - ub[ub.order_hi==0].groupby(["pid","arm"], observed=True)["correct"].mean()).reset_index()
        dU = dsc.loc[dsc.arm=="unrestricted","correct"]; dG = dsc.loc[dsc.arm=="google","correct"]
        tt, pp2 = st.ttest_ind(dU, dG, equal_var=False)
        rej_t += (tt < 0) and (pp2/2 < 0.05)
        # (ii) the pipeline's confirmatory engine
        try:
            gee = sap.fit_bloom_gee(ub)
            b5, se5, p5 = sap.h4a_from_gee(gee)
        except Exception:
            fail += 1; continue
        rej_gee += (p5 < ALPHA)
        if extra == 10: keep.append((b5, se5))
    n = R - fail; pt, pg = rej_t/R, rej_gee/n
    if extra == 10: BLOOM_KEEP["pairs"] = keep
    lab = f"extra={extra}pp"
    vt = v_size(pt,0.05,rate_mcse(pt,R)) if extra==0 else v_target(pt,tgt,rate_mcse(pt,R))
    record("2 Power", f"H4a {lab}: script t-test (DGP replication)", tgt, round(pt,4), round(rate_mcse(pt,R),4), vt)
    vg = v_size(pg,0.05,rate_mcse(pg,n)) if extra==0 else v_target(pg,tgt,rate_mcse(pg,n),tol=0.05)
    record("2 Power", f"H4a {lab}: pipeline GEE (same reps)", tgt if extra else 0.05, round(pg,4), round(rate_mcse(pg,n),4), vg,
           note="pre-registered figure was computed under the t-test, not the GEE" if extra else "")
    if extra and abs(pg-pt) > 0.05:
        DISCREPANCIES.append(f"H4a power at extra={extra}pp: script t-test {pt:.3f} vs pipeline GEE {pg:.3f} "
                             f"— appendix_power.tex quotes the t-test numbers but the SAP's test is the GEE/GLMM.")
TIMES["C2bloom"] = time.time()-t0; print(f"[C2 Bloom done in {TIMES['C2bloom']:.0f}s]")

  [2 Power] H4a extra=0pp: script t-test (DGP replication): nominal=0.05  empirical=0.0367  MCSE=0.0077  -> PASS
  [2 Power] H4a extra=0pp: pipeline GEE (same reps): nominal=0.05  empirical=0.0133  MCSE=0.0047  -> PASS (conservative)


  [2 Power] H4a extra=5pp: script t-test (DGP replication): nominal=0.267  empirical=0.2833  MCSE=0.0184  -> PASS (within MC error)
  [2 Power] H4a extra=5pp: pipeline GEE (same reps): nominal=0.267  empirical=0.16  MCSE=0.015  -> FLAG (diverges)  (pre-registered figure was computed under the t-test, not the GEE)


  [2 Power] H4a extra=10pp: script t-test (DGP replication): nominal=0.695  empirical=0.6683  MCSE=0.0192  -> PASS (within MC error)
  [2 Power] H4a extra=10pp: pipeline GEE (same reps): nominal=0.695  empirical=0.4967  MCSE=0.0204  -> FLAG (diverges)  (pre-registered figure was computed under the t-test, not the GEE)


  [2 Power] H4a extra=15pp: script t-test (DGP replication): nominal=0.956  empirical=0.9667  MCSE=0.0073  -> PASS (within MC error)
  [2 Power] H4a extra=15pp: pipeline GEE (same reps): nominal=0.956  empirical=0.8433  MCSE=0.0148  -> FLAG (diverges)  (pre-registered figure was computed under the t-test, not the GEE)
[C2 Bloom done in 368s]


## Check 3 — Non-inferiority operating characteristics (R = 800 per scenario)
All scenarios put the Unrestricted deficit at 17 pp so gate 1 opens with probability ≈ 1 and the
NI step is actually reached. Scenarios: **(i)** true Socratic−Google = 0 — the D6 power claim.
The same replicates evaluate BOTH decision rules: the coded rule (90% CI lower > −Δ, one-sided
α=.05; analytic expectation ≈ 0.87) and the power script's computed rule (α=.025 / 95% CI;
its 0.792). **(ii)** deficit −12 pp (beyond Δ): declaration must be rare. **(iii)** boundary
−8.5 pp: the NI test's true size (target ≤ .05). **(iv)** noncompliance: 20% of the Socratic
arm never engages and earns no complier benefit (compliers −7 pp, noncompliers 0) — measures
how often the pre-registered ITT ∧ per-protocol conjunction changes the ITT-only verdict.

In [7]:
t0 = time.time(); rng = np.random.default_rng(SEED + 4)
for tag, soc_eff, tgt90, tgt95 in [("(i) delta_true=0", 0.0, 0.87, 0.792),
                                   ("(ii) deficit 12pp", -0.12, None, None),
                                   ("(iii) boundary 8.5pp", -0.085, 0.05, None)]:
    R = REPS["C3"]; fail = 0; ni90 = ni95 = reach = 0
    for r in range(R):
        acc_pw = sim_acc_trial(rng, b_soc=soc_eff, b_unr=-0.17)
        try:
            gk = sap.gatekeeper(acc_pw, set(acc_pw.pid), alpha=ALPHA, delta=DELTA)
        except Exception:
            fail += 1; continue
        reach += gk["rej_h1a"]
        ni90 += gk["ni_declared"]                       # coded rule (90% CI, ITT&PP)
        lo95 = gk["b1"] - 1.959963985 * gk["se1"]       # counterfactual 95%-CI rule
        ni95 += bool(gk["rej_h1a"] and lo95 > -DELTA)
    n = R - fail; p90, p95, pr = ni90/n, ni95/n, reach/n
    print(f"{tag}: gate-1 open rate {pr:.3f}")
    if tag.startswith("(i)"):
        record("3 NI", f"{tag}: coded rule 90%CI (SAP/v3)", tgt90, round(p90,4), round(rate_mcse(p90,n),4), v_target(p90,tgt90,rate_mcse(p90,n)), note="analytic ~0.87 at alpha=.05")
        record("3 NI", f"{tag}: script rule 95%CI (alpha=.025)", tgt95, round(p95,4), round(rate_mcse(p95,n),4), v_target(p95,tgt95,rate_mcse(p95,n)), note="reproduces the 0.792 the script computed")
        DISCREPANCIES.append(
            f"NI POWER RULE MISMATCH: at delta_true=0 the coded 90%-CI rule declares NI in {p90:.3f} of trials; "
            f"the alpha=.025/95%-CI rule the power script actually computed gives {p95:.3f} (its published 0.792). "
            "power_analysis_v2.py line 139 uses alpha=0.025 while its caption, appendix_power.tex l.129/144, the SAP "
            "and v3 specify the 90% CI (one-sided .05). AUTHOR DECISION: fix the script/caption to alpha=.05 "
            "(NI power becomes ~0.87 — no design change), or adopt the stricter 95% rule in the SAP.")
    elif tag.startswith("(ii)"):
        record("3 NI", f"{tag}: false declaration rate", "low (<~.03)", round(p90,4), round(rate_mcse(max(p90,1e-4),n),4), "PASS" if p90 < 0.05 else "FAIL")
    else:
        record("3 NI", f"{tag}: NI size at the margin", 0.05, round(p90,4), round(rate_mcse(p90,n),4), v_size(p90,0.05,rate_mcse(p90,n)))
TIMES["C3"] = time.time()-t0; print(f"[C3 i-iii done in {TIMES['C3']:.0f}s]")

(i) delta_true=0: gate-1 open rate 1.000
  [3 NI] (i) delta_true=0: coded rule 90%CI (SAP/v3): nominal=0.87  empirical=0.8967  MCSE=0.0124  -> PASS (within MC error)  (analytic ~0.87 at alpha=.05)
  [3 NI] (i) delta_true=0: script rule 95%CI (alpha=.025): nominal=0.792  empirical=0.805  MCSE=0.0162  -> PASS (within MC error)  (reproduces the 0.792 the script computed)


(ii) deficit 12pp: gate-1 open rate 1.000
  [3 NI] (ii) deficit 12pp: false declaration rate: nominal=low (<~.03)  empirical=0.0033  MCSE=0.0024  -> PASS


(iii) boundary 8.5pp: gate-1 open rate 1.000
  [3 NI] (iii) boundary 8.5pp: NI size at the margin: nominal=0.05  empirical=0.0633  MCSE=0.0099  -> PASS
[C3 i-iii done in 564s]


In [8]:
# (iv) noncompliance: does the ITT AND per-protocol conjunction ever bite?
t0 = time.time(); rng = np.random.default_rng(SEED + 5)
R = REPS["C3IV"]; fail = 0; itt = pp = both = disagree = 0
NC = int(0.20 * N_ARM)                       # 20% Socratic non-engagers
for r in range(R):
    soc_vec = np.full(N_ARM, -0.07)          # compliers: -7pp (inside the margin)
    idx_nc = rng.choice(N_ARM, NC, replace=False)
    soc_vec[idx_nc] = 0.0                    # non-engagers: no effect
    acc_pw = sim_acc_trial(rng, b_soc=soc_vec, b_unr=-0.17)
    engaged_pids = set(np.arange(3*N_ARM)) - set(N_ARM + idx_nc)   # PP excludes non-engagers
    try:
        gk = sap.gatekeeper(acc_pw, engaged_pids, alpha=ALPHA, delta=DELTA)
    except Exception:
        fail += 1; continue
    i_ok = bool(gk["rej_h1a"] and gk["ni_itt"]); p_ok = bool(gk["rej_h1a"] and (gk["ni_pp"] is True))
    itt += i_ok; pp += p_ok; both += gk["ni_declared"]; disagree += (i_ok != p_ok)
n = R - fail
print(f"reps={n} (failures={fail}) | P(ITT pass)={itt/n:.3f}  P(PP pass)={pp/n:.3f}  "
      f"P(conjunction)={both/n:.3f}  P(ITT vs PP disagree)={disagree/n:.3f}")
record("3 NI", "(iv) noncompliance: conjunction stricter than ITT-only", "conj <= ITT, disagree > 0",
       f"conj={both/n:.3f} vs ITT={itt/n:.3f}; disagree={disagree/n:.3f}", round(rate_mcse(disagree/n,n),4),
       "PASS" if (both <= itt and disagree > 0) else "FAIL (conjunction vacuous)")
TIMES["C3IV"] = time.time()-t0; print(f"[C3 iv done in {TIMES['C3IV']:.0f}s]")

reps=500 (failures=0) | P(ITT pass)=0.296  P(PP pass)=0.156  P(conjunction)=0.150  P(ITT vs PP disagree)=0.152
  [3 NI] (iv) noncompliance: conjunction stricter than ITT-only: nominal=conj <= ITT, disagree > 0  empirical=conj=0.150 vs ITT=0.296; disagree=0.152  MCSE=0.0161  -> PASS
[C3 iv done in 330s]


## Check 4 — The persistence estimand (D9-R1), demonstrated not asserted
**(a)** Deficit constant at −10 pp in both waves (β₅ = 0): the simple effect β₂+β₅ must affirm
persistence (target ≈ 0.948, the script's 10 pp power row), while BOTH interaction-based
readings fail — testing β₅<0 almost never rejects (there is no *growth*), and the
"non-significant β₅ ⇒ persists" reading is exposed in (b). **(b)** Full decay (−10 pp
immediate, 0 delayed, β₅ = +10 pp): the simple effect must NOT affirm (size ≤ .05), while the
"null-β₅ ⇒ persists" reading wrongly affirms whenever the interaction test misses the decay.
**(c)** −8 pp constant deficit + 30% random attrition: the script's analytic
MMRM-recovery approximation promises 0.743. R = 800/800/600.

In [9]:
t0 = time.time(); rng = np.random.default_rng(SEED + 6)
res_rows = {}
for tag, bu, buw, drop, R in [("(a) constant -10pp", -0.10, 0.0, None, REPS["C4"]),
                              ("(b) full decay",     -0.10, +0.10, None, REPS["C4"]),
                              ("(c) -8pp + 30% attrition", -0.08, 0.0, 0.30, REPS["C4C"])]:
    fail = 0; rej_se = rej_b5neg = affirm_nullb5 = 0
    for r in range(R):
        acc_pw = sim_acc_trial(rng, b_unr=bu, b_unr_w=buw, drop_delayed=drop)
        try:
            res = sap.fit_primary_acc(acc_pw)
            est, se_, p5 = sap.h5a_simple_effect(res)          # D9-R1 estimand
            b5, s5 = sap.lincom(res, {"unrestricted:wave_d": 1})
        except Exception:
            fail += 1; continue
        rej_se += (p5 < ALPHA)
        rej_b5neg += (st.norm.cdf(b5 / s5) < ALPHA)            # WRONG reading 1: beta5<0
        affirm_nullb5 += (abs(b5 / s5) < 1.959963985)          # WRONG reading 2: n.s. beta5 => "persists"
    n = R - fail
    res_rows[tag] = (rej_se/n, rej_b5neg/n, affirm_nullb5/n, n)

p, pb, pn, n = res_rows["(a) constant -10pp"]
record("4 Persist", "(a) constant deficit: simple-effect power", 0.948, round(p,4), round(rate_mcse(p,n),4), v_target(p,0.948,rate_mcse(p,n)))
record("4 Persist", "(a) WRONG reading beta5<0 affirms", "~0 (cannot see persistence)", round(pb,4), round(rate_mcse(max(pb,1e-4),n),4),
       "PASS (demonstrates D9-R1)" if pb < 0.10 else "FAIL", note="interaction is null when the deficit persists unchanged")
p, pb, pn, n = res_rows["(b) full decay"]
record("4 Persist", "(b) full decay: simple-effect size", 0.05, round(p,4), round(rate_mcse(p,n),4), v_size(p,0.05,rate_mcse(p,n)))
record("4 Persist", "(b) WRONG reading n.s.-beta5 => persists", "high false-affirm rate", round(pn,4), round(rate_mcse(pn,n),4),
       "PASS (demonstrates D9-R1)" if pn > 0.15 else "FLAG", note="absence-of-evidence fallacy quantified")
p, pb, pn, n = res_rows["(c) -8pp + 30% attrition"]
record("4 Persist", "(c) 8pp deficit, 30% attrition: simple-effect power", 0.743, round(p,4), round(rate_mcse(p,n),4), v_target(p,0.743,rate_mcse(p,n)),
       note="validates the script's n_eff = n(1-a)+n*a*rho^2 approximation")
TIMES["C4"] = time.time()-t0; print(f"[C4 done in {TIMES['C4']:.0f}s]")

  [4 Persist] (a) constant deficit: simple-effect power: nominal=0.948  empirical=0.9483  MCSE=0.009  -> PASS (within MC error)
  [4 Persist] (a) WRONG reading beta5<0 affirms: nominal=~0 (cannot see persistence)  empirical=0.0417  MCSE=0.0082  -> PASS (demonstrates D9-R1)  (interaction is null when the deficit persists unchanged)
  [4 Persist] (b) full decay: simple-effect size: nominal=0.05  empirical=0.0583  MCSE=0.0096  -> PASS
  [4 Persist] (b) WRONG reading n.s.-beta5 => persists: nominal=high false-affirm rate  empirical=0.2133  MCSE=0.0167  -> PASS (demonstrates D9-R1)  (absence-of-evidence fallacy quantified)
  [4 Persist] (c) 8pp deficit, 30% attrition: simple-effect power: nominal=0.743  empirical=0.708  MCSE=0.0203  -> PASS (within MC error)  (validates the script's n_eff = n(1-a)+n*a*rho^2 approximation)
[C4 done in 567s]


## Check 5 — Cox model recovery, clustering SEs, and the MDHR claim
True immediate log-HR θ₂ = log 0.75, delayed θ₂+θ₄ = log 0.90 (θ₄ = log 1.2) on the
correct-cause hazard. **(5a)** No frailty (R = 800): the fitted θ̂'s must be unbiased for the
DGP truth and the participant-clustered 95% CIs must cover ≈ .95 (with independent items the
robust estimator must not distort). **(5b)** Frailty σ_b = 0.6 (R = 800): the Cox coefficient
now targets the *marginally attenuated* population parameter (non-collapsibility), so truth is
defined as the **plim** — measured once from a mega-trial (2 500/arm) — and coverage is
evaluated against it, clustered vs naive SEs side by side. **(5c)** The script's minimum
detectable HR: immediate wave, two arms, true correct-cause HR = 1.259, frailty as in 5b
(R = 600) — the Schoenfeld-with-design-effect bound promises 0.80 and calls itself an upper
bound; the empirical number prices that caveat.

In [10]:
t0 = time.time(); rng = np.random.default_rng(SEED + 7)
TH2, TH4 = np.log(0.75), np.log(0.90) - np.log(0.75)
R = REPS["C5A"]; fail = 0
e2, e4, cov2, cov4 = [], [], 0, 0
for r in range(R):
    surv = sim_cox_trial(rng, sigma_b=0.0)
    try:
        cph = sap.fit_cox_st(surv)
    except Exception:
        fail += 1; continue
    b, s = sap.cox_lincom(cph, {"unrestricted": 1}); e2.append(b); cov2 += (abs(b-TH2) <= 1.959963985*s)
    b, s = sap.cox_lincom(cph, {"unr_wave": 1});     e4.append(b); cov4 += (abs(b-TH4) <= 1.959963985*s)
n = R - fail
record("5 Cox", "(5a) no-frailty: theta2 bias", round(TH2,4), round(float(np.mean(e2)),4), round(float(np.std(e2,ddof=1)/np.sqrt(n)),4),
       "PASS" if abs(np.mean(e2)-TH2) < 3*np.std(e2,ddof=1)/np.sqrt(n) else "FAIL")
record("5 Cox", "(5a) no-frailty: theta4 bias", round(TH4,4), round(float(np.mean(e4)),4), round(float(np.std(e4,ddof=1)/np.sqrt(n)),4),
       "PASS" if abs(np.mean(e4)-TH4) < 3*np.std(e4,ddof=1)/np.sqrt(n) else "FAIL")
for lab, cv in [("theta2", cov2/n), ("theta4", cov4/n)]:
    record("5 Cox", f"(5a) clustered 95% CI coverage, {lab}", 0.95, round(cv,4), round(rate_mcse(cv,n),4), v_cover(cv, rate_mcse(cv,n)))
TIMES["C5A"] = time.time()-t0; print(f"[C5a done in {TIMES['C5A']:.0f}s]")

  [5 Cox] (5a) no-frailty: theta2 bias: nominal=-0.2877  empirical=-0.2898  MCSE=0.0027  -> PASS
  [5 Cox] (5a) no-frailty: theta4 bias: nominal=0.1823  empirical=0.1813  MCSE=0.004  -> PASS
  [5 Cox] (5a) clustered 95% CI coverage, theta2: nominal=0.95  empirical=0.954  MCSE=0.0094  -> PASS
  [5 Cox] (5a) clustered 95% CI coverage, theta4: nominal=0.95  empirical=0.94  MCSE=0.0106  -> PASS
[C5a done in 641s]


In [11]:
# (5b) frailty: plim as the estimand; clustered vs naive coverage
t0 = time.time(); rng = np.random.default_rng(SEED + 8)
SIGB = 0.6
mega = sim_cox_trial(rng, n_arm=1500, sigma_b=SIGB)
# achieved event-status ICC (one-way ANOVA estimator on the immediate wave)
mi = mega[mega.wave_d == 0]
grp = mi.groupby("pid")["event"].agg(["mean", "count"])
pbar = mi["event"].mean(); m = grp["count"].mean()
msb = (grp["count"] * (grp["mean"] - pbar) ** 2).sum() / (len(grp) - 1)
msw = (mi.groupby("pid")["event"].var(ddof=1) * (grp["count"] - 1)).sum() / (grp["count"] - 1).sum()
icc = (msb - msw) / (msb + (m - 1) * msw)
cph_mega = sap.fit_cox_st(mega)
PL2, _ = sap.cox_lincom(cph_mega, {"unrestricted": 1})
PL4, _ = sap.cox_lincom(cph_mega, {"unr_wave": 1})
print(f"plim (2500/arm): theta2={PL2:+.4f} (conditional truth {TH2:+.4f}, attenuation {PL2-TH2:+.4f}) | "
      f"theta4={PL4:+.4f} (truth {TH4:+.4f}) | achieved event ICC={icc:.3f}")
if abs(PL2 - TH2) > 0.02:
    DISCREPANCIES.append(
        f"Cox non-collapsibility: with participant heterogeneity (sigma_b={SIGB}, event ICC~{icc:.2f}) the "
        f"marginal Cox coefficient is attenuated (plim {PL2:+.3f} vs conditional {TH2:+.3f}). The SAP's HR "
        "estimand is the marginal one — fine — but MDHR/power statements calibrated to conditional HRs are "
        "optimistic by this factor.")
R = REPS["C5B"]; fail = 0; est2 = []; covc2 = covn2 = covc4 = covn4 = 0
for r in range(R):
    surv = sim_cox_trial(rng, sigma_b=SIGB)
    try:
        cph_c = sap.fit_cox_st(surv)                       # clustered (the pipeline)
        cph_n = sap.fit_cox_st(surv, cluster=None)         # naive comparison
    except Exception:
        fail += 1; continue
    b, s = sap.cox_lincom(cph_c, {"unrestricted": 1}); est2.append(b)
    covc2 += (abs(b - PL2) <= 1.959963985 * s)
    b4, s4 = sap.cox_lincom(cph_c, {"unr_wave": 1}); covc4 += (abs(b4 - PL4) <= 1.959963985 * s4)
    bn, sn = sap.cox_lincom(cph_n, {"unrestricted": 1}); covn2 += (abs(bn - PL2) <= 1.959963985 * sn)
    bn4, sn4 = sap.cox_lincom(cph_n, {"unr_wave": 1}); covn4 += (abs(bn4 - PL4) <= 1.959963985 * sn4)
n = R - fail
record("5 Cox", "(5b) frailty: theta2_hat centres on plim", round(PL2,4), round(float(np.mean(est2)),4),
       round(float(np.std(est2,ddof=1)/np.sqrt(n)),4),
       "PASS" if abs(np.mean(est2)-PL2) < 3*np.std(est2,ddof=1)/np.sqrt(n) else "FAIL")
for lab, cc, cn in [("theta2", covc2/n, covn2/n), ("theta4", covc4/n, covn4/n)]:
    record("5 Cox", f"(5b) clustered 95% coverage, {lab}", 0.95, round(cc,4), round(rate_mcse(cc,n),4), v_cover(cc, rate_mcse(cc,n)))
    record("5 Cox", f"(5b) NAIVE 95% coverage, {lab} (comparison)", 0.95, round(cn,4), round(rate_mcse(cn,n),4),
           "expected undercoverage" if cn < 0.93 else "note: clustering mild for this term")
DISCREPANCIES.append(
    "PIPELINE DEFECT FOUND AND FIXED DURING VALIDATION (most consequential): v2/v3 built every Cox "
    "contrast from cph.variance_matrix_, which lifelines leaves as the NAIVE covariance even when "
    "cluster_col is set (robust SEs live only in standard_errors_, per-coefficient). Under realistic "
    "within-participant clustering the success-time contrast SEs were ~40-60% too small (pre-fix 95% "
    "CI coverage ~0.70 in this check). sap_estimators.fit_cox_st now attaches the full Lin-Wei "
    "sandwich (diagonal matched exactly to lifelines' published robust SEs) and cox_lincom uses it; "
    "v3's two inline Cox fits (productivity, R1) were re-pointed at fit_cox_st. All success-time "
    "CIs and p-values widen accordingly. ACTION: none for the SAP text (it already promises Lin 1989 "
    "cluster-robust inference — the CODE now actually delivers it), but v3 outputs must be re-run.")
TIMES["C5B"] = time.time()-t0; print(f"[C5b done in {TIMES['C5B']:.0f}s]")

plim (2500/arm): theta2=-0.2469 (conditional truth -0.2877, attenuation +0.0408) | theta4=+0.1469 (truth +0.1823) | achieved event ICC=0.083


  [5 Cox] (5b) frailty: theta2_hat centres on plim: nominal=-0.2469  empirical=-0.2465  MCSE=0.0055  -> PASS
  [5 Cox] (5b) clustered 95% coverage, theta2: nominal=0.95  empirical=0.9475  MCSE=0.0112  -> PASS
  [5 Cox] (5b) NAIVE 95% coverage, theta2 (comparison): nominal=0.95  empirical=0.725  MCSE=0.0223  -> expected undercoverage
  [5 Cox] (5b) clustered 95% coverage, theta4: nominal=0.95  empirical=0.9525  MCSE=0.0106  -> PASS
  [5 Cox] (5b) NAIVE 95% coverage, theta4 (comparison): nominal=0.95  empirical=0.98  MCSE=0.007  -> note: clustering mild for this term
[C5b done in 843s]


In [12]:
# (5c) the script's MDHR: HR=1.259 @ 80% (Schoenfeld bound, DE-deflated) — empirical price
t0 = time.time(); rng = np.random.default_rng(SEED + 9)
R = REPS["C5C"]; fail = 0; rej = 0
for r in range(R):
    surv = sim_cox_trial(rng, th_unr=(np.log(1.259), np.log(1.259)), sigma_b=0.6,
                         waves=(0,), arms=3)
    surv = surv[surv.arm.isin(["google", "unrestricted"])]
    try:
        cph = sap.fit_cox_st(surv, formula="unrestricted + minivlat_c + aiuse_c", strata=None)
    except Exception:
        fail += 1; continue
    b, s = sap.cox_lincom(cph, {"unrestricted": 1})
    rej += (st.norm.sf(b / s) < ALPHA)          # one-sided: faster correct production
n = R - fail; p = rej/n
record("5 Cox", "(5c) power at the claimed MDHR 1.259 (frailty ICC~0.1)", 0.80, round(p,4), round(rate_mcse(p,n),4),
       v_target(p, 0.80, rate_mcse(p,n), tol=0.05),
       note="script itself flags Schoenfeld as an upper bound; shortfall = the price of that caveat + non-collapsibility")
if p < 0.75:
    DISCREPANCIES.append(
        f"Cox MDHR: empirical power at conditional HR=1.259 is {p:.3f} vs the Schoenfeld-bound 0.80 "
        "(power_results_v2.txt l.68). Sources: marginal attenuation under frailty + the bound's own optimism. "
        "The appendix already calls it an upper bound; consider quoting the empirical clustered figure instead.")
TIMES["C5C"] = time.time()-t0; print(f"[C5c done in {TIMES['C5C']:.0f}s]")

  [5 Cox] (5c) power at the claimed MDHR 1.259 (frailty ICC~0.1): nominal=0.8  empirical=0.63  MCSE=0.0216  -> FLAG (diverges)  (script itself flags Schoenfeld as an upper bound; shortfall = the price of that caveat + non-collapsibility)
[C5c done in 113s]


## Check 6 — Bloom engines: GEE calibration vs the VB GLMM's anticonservatism
Fixed DGP at extra = 10 pp. The GEE's estimand is the **marginal** (population-averaged)
log-odds interaction — computed here by 2-million-draw numerical integration of the DGP; the
VB GLMM's estimand is the **conditional** one (the DGP's cell-logit DiD). Each engine is judged
against *its own* truth: bias, SE calibration (mean reported SE ÷ SD of estimates across
replicates — the honest-uncertainty ratio), and CI coverage. GEE reuses the 600 stored fits
from Check 2; the VB model (v3's corroboration code, copied verbatim) runs R = 200 fresh fits.

In [13]:
t0 = time.time()
from scipy.special import expit, logit
th = np.random.default_rng(123).normal(0, 0.7, 2_000_000)
def marg(p): return float(np.mean(expit(logit(p) + th)))
pU = (0.75, 0.50 - 0.05 - 0.10)
mGL, mGH, mUL, mUH = marg(0.80), marg(0.50), marg(pU[0]), marg(pU[1])
TRUTH_MARG = (logit(mUH) - logit(mUL)) - (logit(mGH) - logit(mGL))
TRUTH_COND = (logit(pU[1]) - logit(pU[0])) - (logit(0.50) - logit(0.80))
print(f"interaction truth: marginal (GEE estimand) {TRUTH_MARG:+.4f} | conditional (GLMM estimand) {TRUTH_COND:+.4f}")

pairs = BLOOM_KEEP["pairs"]; est = np.array([p[0] for p in pairs]); ses = np.array([p[1] for p in pairs])
n = len(est)
bias = float(est.mean() - TRUTH_MARG); calib = float(ses.mean() / est.std(ddof=1))
cov = float(np.mean(np.abs(est - TRUTH_MARG) <= 1.959963985 * ses))
record("6 Bloom", "GEE bias vs marginal truth", round(TRUTH_MARG,4), round(float(est.mean()),4),
       round(float(est.std(ddof=1)/np.sqrt(n)),4), "PASS" if abs(bias) < 3*est.std(ddof=1)/np.sqrt(n) else "FAIL")
record("6 Bloom", "GEE SE calibration (mean SE / SD est)", 1.0, round(calib,3), round(1/np.sqrt(2*(n-1)),3),
       "PASS" if abs(calib-1) < 0.07 else "FLAG")
record("6 Bloom", "GEE 95% CI coverage", 0.95, round(cov,4), round(rate_mcse(cov,n),4), v_cover(cov, rate_mcse(cov,n)))
TIMES["C6GEE"] = time.time()-t0; print(f"[C6 GEE done in {TIMES['C6GEE']:.0f}s]")

interaction truth: marginal (GEE estimand) -0.2938 | conditional (GLMM estimand) -0.3314


  [6 Bloom] GEE bias vs marginal truth: nominal=-0.2938  empirical=-0.2883  MCSE=0.008  -> PASS
  [6 Bloom] GEE SE calibration (mean SE / SD est): nominal=1.0  empirical=0.937  MCSE=0.029  -> PASS
  [6 Bloom] GEE 95% CI coverage: nominal=0.95  empirical=0.935  MCSE=0.0101  -> PASS
[C6 GEE done in 0s]


In [14]:
# VB GLMM (v3's corroboration model, copied verbatim from the analysis notebook)
t0 = time.time(); rng = np.random.default_rng(SEED + 10)
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM
R = REPS["C6VB"]; fail = 0; means, sds = [], []
for r in range(R):
    ub = sim_bloom_trial(rng, 10)
    try:
        bm = BinomialBayesMixedGLM.from_formula(
            "correct ~ (socratic + unrestricted) * order_hi + minivlat_c + aiuse_c",
            {"pid": "0 + C(pid)", "item": "0 + C(item_id)"}, ub).fit_vb()
        k = bm.model.exog_names.index("unrestricted:order_hi")
        m_k, s_k = float(bm.fe_mean[k]), float(bm.fe_sd[k])   # fe_sd = VB posterior SD
    except Exception:
        fail += 1; continue
    means.append(m_k); sds.append(s_k)
n = R - fail; means, sds = np.array(means), np.array(sds)
bias = float(means.mean() - TRUTH_COND); calib = float(sds.mean() / means.std(ddof=1))
cov = float(np.mean(np.abs(means - TRUTH_COND) <= 1.959963985 * sds))
record("6 Bloom", "VB GLMM bias vs conditional truth", round(TRUTH_COND,4), round(float(means.mean()),4),
       round(float(means.std(ddof=1)/np.sqrt(n)),4), "PASS" if abs(bias) < max(3*means.std(ddof=1)/np.sqrt(n), .05) else "FLAG",
       note=f"failures={fail}")
record("6 Bloom", "VB post-SD calibration (mean postSD / SD est)", 1.0, round(calib,3), round(1/np.sqrt(2*(n-1)),3),
       "FAIL (anticonservative)" if calib < 0.9 else ("PASS" if calib < 1.1 else "note: conservative"))
record("6 Bloom", "VB 95% credible-interval coverage of truth", 0.95, round(cov,4), round(rate_mcse(cov,n),4), v_cover(cov, rate_mcse(cov,n)))
if calib < 0.9 or cov < 0.9:
    DISCREPANCIES.append(
        f"VB GLMM anticonservatism CONFIRMED: posterior SDs are ~{calib:.2f}x the true sampling SD and interval "
        f"coverage is {cov:.2f}. The SAP names BinomialBayesMixedGLM as the eq:bloom engine; v3 already demotes "
        "it to corroboration behind the GEE — the SAP text should adopt that demotion explicitly.")
TIMES["C6VB"] = time.time()-t0; print(f"[C6 VB done in {TIMES['C6VB']:.0f}s]")

  [6 Bloom] VB GLMM bias vs conditional truth: nominal=-0.3314  empirical=-0.3188  MCSE=0.0213  -> PASS  (failures=0)
  [6 Bloom] VB post-SD calibration (mean postSD / SD est): nominal=1.0  empirical=0.445  MCSE=0.071  -> FAIL (anticonservative)
  [6 Bloom] VB 95% credible-interval coverage of truth: nominal=0.95  empirical=0.65  MCSE=0.0477  -> FAIL
[C6 VB done in 209s]


## Check 7 — Missing data: MAR vs MNAR, and whether Lee bounds actually bracket
Truth: constant −8 pp Unrestricted deficit at both waves. **MAR** (R = 400, incl. MICE m=10):
delayed-wave response depends on the *observed* wave-1 score and covariate with arm-specific
intercepts (control retains ~90%, LLM arms ~72%) — the MMRM (which models wave 1 jointly) and
MICE must stay unbiased while the naive complete-case delayed contrast shifts. **MNAR**
(R = 800): response = 1{V_i > c_arm} with V_i loading on the *unobserved* delayed shock
(monotone across arms, so Lee's assumption holds and the always-responder effect equals −8 pp
by effect homogeneity). Complete-case and MMRM are now both biased; the two-sided
`sap.lee_bounds` must bracket the truth.

In [15]:
t0 = time.time(); rng = np.random.default_rng(SEED + 12)
from statsmodels.imputation import mice as sm_mice
TRUTH = -0.08
def mar_drop(y_i, y_d, u, eps_d, c1, rr):
    a0 = np.where(ARMS == "google", 2.2, 0.9)          # arm-specific retention
    pr = 1/(1+np.exp(-(a0 + 2.0*(y_i - P0)/SD_ACC + 0.3*c1)))
    return rr.random(NTOT) < pr
R = REPS["C7MAR"]; fail = 0; e_mm, e_cc, e_mi = [], [], []
for r in range(R):
    np.random.seed(SEED + 12000 + r)                    # MICEData uses global numpy state
    acc_pw = sim_acc_trial(rng, b_unr=TRUTH, b_unr_w=0.0, drop_delayed=mar_drop)
    try:
        res = sap.fit_primary_acc(acc_pw)
        e_mm.append(sap.h5a_simple_effect(res)[0])
        dl = acc_pw[acc_pw.wave_d == 1]
        o = smf.ols("acc ~ socratic + unrestricted + minivlat_c + aiuse_c", dl).fit()
        e_cc.append(float(o.params["unrestricted"]))
        wide = (acc_pw.pivot_table(index="pid", columns="wave_d", values="acc")
                .rename(columns={0: "immediate", 1: "delayed"}).reset_index()
                .merge(acc_pw[acc_pw.wave_d == 0][["pid","socratic","unrestricted","minivlat_c","aiuse_c"]], on="pid"))
        imp = sm_mice.MICEData(wide.drop(columns="pid").astype(float))
        fit = sm_mice.MICE("delayed ~ socratic + unrestricted + minivlat_c + aiuse_c", sm.OLS, imp).fit(10, 10)
        e_mi.append(float(np.asarray(fit.params)[2]))   # [Intercept, socratic, unrestricted, ...]
    except Exception:
        fail += 1; continue
n = R - fail
for lab, arr, expect_ok in [("MMRM delayed simple effect", e_mm, True),
                            ("naive complete-case OLS", e_cc, False),
                            ("MICE (m=10) pooled", e_mi, True)]:
    a = np.array(arr); mcse = float(a.std(ddof=1)/np.sqrt(len(a))); bias = float(a.mean() - TRUTH)
    if expect_ok:
        vd = "PASS" if abs(bias) < max(3*mcse, 0.006) else "FAIL (biased under MAR)"
    else:
        vd = "PASS (bias demonstrated)" if abs(bias) > 3*mcse else "note: selection too weak"
    record("7 Missing", f"MAR: {lab} (truth -0.080)", -0.08, round(float(a.mean()),4), round(mcse,4), vd,
           note=f"bias={bias:+.4f}")
TIMES["C7MAR"] = time.time()-t0; print(f"[C7 MAR done in {TIMES['C7MAR']:.0f}s] failures={fail}")

  [7 Missing] MAR: MMRM delayed simple effect (truth -0.080): nominal=-0.08  empirical=-0.0788  MCSE=0.0021  -> PASS  (bias=+0.0012)
  [7 Missing] MAR: naive complete-case OLS (truth -0.080): nominal=-0.08  empirical=-0.0643  MCSE=0.0021  -> PASS (bias demonstrated)  (bias=+0.0157)
  [7 Missing] MAR: MICE (m=10) pooled (truth -0.080): nominal=-0.08  empirical=-0.0596  MCSE=0.0018  -> FAIL (biased under MAR)  (bias=+0.0204)
[C7 MAR done in 723s] failures=0


In [16]:
# MNAR + Lee bounds
t0 = time.time(); rng = np.random.default_rng(SEED + 13)
sdV = np.sqrt(SIG_U**2 + 0.25 * SIG_E**2)
cG, cU = st.norm.ppf(0.10) * sdV, st.norm.ppf(0.30) * sdV     # retain 90% / 70%
def mnar_drop(y_i, y_d, u, eps_d, c1, rr):
    V = u + 0.5 * eps_d                                        # loads on the UNOBSERVED delayed shock
    thr = np.where(ARMS == "google", cG, np.where(ARMS == "unrestricted", cU, cG))
    return V > thr
# Population Lee bounds: sap.lee_bounds applied to a mega-trial (no model fits, cheap).
# The correct asymptotic claim is that the POPULATION bounds bracket the always-responder
# effect; finite-sample bounds are noisy estimates of those population bounds (a truth
# sitting near a population bound edge is bracketed by sample bounds only ~50% of the
# time — that is geometry, not failure).
NA_MEGA = 40000
c1m = rng.normal(0, 1, 3*NA_MEGA); c2m = rng.normal(0, 1, 3*NA_MEGA)
um = rng.normal(0, SIG_U, 3*NA_MEGA); edm = rng.normal(0, SIG_E, 3*NA_MEGA)
armm = np.repeat(["google","socratic","unrestricted"], NA_MEGA)
ydm = P0 + 0.03 + TRUTH*(armm=="unrestricted") + G_LOAD*c1m + G_LOAD*c2m + um + edm
Vm = um + 0.5*edm
thrm = np.where(armm=="google", cG, np.where(armm=="unrestricted", cU, cG))
keepm = Vm > thrm
POP_LO, POP_HI = sap.lee_bounds(ydm[keepm & (armm=="unrestricted")], NA_MEGA,
                                ydm[keepm & (armm=="google")], NA_MEGA)
print(f"population Lee bounds (mega, n/arm={NA_MEGA}): [{POP_LO:+.4f}, {POP_HI:+.4f}] — truth {TRUTH:+.3f}")
record("7 Missing", "MNAR: POPULATION Lee bounds bracket the truth", f"lo <= {TRUTH} <= hi",
       f"[{POP_LO:+.4f}, {POP_HI:+.4f}]", "-",
       "PASS" if POP_LO - 0.002 <= TRUTH <= POP_HI + 0.002 else "FAIL (bounds logic wrong)")

R = REPS["C7MNAR"]; fail = 0; e_cc, e_mm = [], []; brack = 0; los, his = [], []
for r in range(R):
    acc_pw = sim_acc_trial(rng, b_unr=TRUTH, b_unr_w=0.0, drop_delayed=mnar_drop)
    dl = acc_pw[acc_pw.wave_d == 1]
    try:
        o = smf.ols("acc ~ socratic + unrestricted + minivlat_c + aiuse_c", dl).fit()
        e_cc.append(float(o.params["unrestricted"]))
        res = sap.fit_primary_acc(acc_pw)
        e_mm.append(sap.h5a_simple_effect(res)[0])
        yu = dl.loc[dl.arm == "unrestricted", "acc"].values
        yg = dl.loc[dl.arm == "google", "acc"].values
        lo, hi = sap.lee_bounds(yu, N_ARM, yg, N_ARM)
        brack += (lo - 1e-12 <= TRUTH <= hi + 1e-12); los.append(lo); his.append(hi)
    except Exception:
        fail += 1; continue
n = R - fail
cc, mm = np.array(e_cc), np.array(e_mm)
record("7 Missing", "MNAR: complete-case (must be biased)", -0.08, round(float(cc.mean()),4),
       round(float(cc.std(ddof=1)/np.sqrt(n)),4),
       "PASS (bias demonstrated)" if abs(cc.mean()-TRUTH) > 3*cc.std(ddof=1)/np.sqrt(n) else "note: weak selection",
       note=f"bias={float(cc.mean()-TRUTH):+.4f}")
record("7 Missing", "MNAR: MMRM (biased too — MAR-only tool)", -0.08, round(float(mm.mean()),4),
       round(float(mm.std(ddof=1)/np.sqrt(n)),4),
       "expected bias shown" if abs(mm.mean()-TRUTH) > 3*mm.std(ddof=1)/np.sqrt(n) else "note: small bias",
       note=f"bias={float(mm.mean()-TRUTH):+.4f}")
for lab, arr, pop in [("sample lower bound centres on population lower", los, POP_LO),
                      ("sample upper bound centres on population upper", his, POP_HI)]:
    a = np.array(arr); mcse = float(a.std(ddof=1)/np.sqrt(n))
    record("7 Missing", f"MNAR: {lab}", round(pop,4), round(float(a.mean()),4), round(mcse,4),
           "PASS" if abs(a.mean()-pop) < max(3*mcse, 0.006) else "FAIL")
pb = brack/n
record("7 Missing", "MNAR: per-sample bracket rate (descriptive)", "~0.5 when truth sits at a bound edge",
       round(pb,4), round(rate_mcse(pb,n),4), "note: geometry, see markdown",
       note=f"truth {TRUTH:+.3f} vs pop bounds [{POP_LO:+.3f}, {POP_HI:+.3f}]")
TIMES["C7MNAR"] = time.time()-t0; print(f"[C7 MNAR done in {TIMES['C7MNAR']:.0f}s] failures={fail}")

population Lee bounds (mega, n/arm=40000): [-0.0993, +0.0442] — truth -0.080
  [7 Missing] MNAR: POPULATION Lee bounds bracket the truth: nominal=lo <= -0.08 <= hi  empirical=[-0.0993, +0.0442]  MCSE=-  -> PASS


  [7 Missing] MNAR: complete-case (must be biased): nominal=-0.08  empirical=-0.0303  MCSE=0.0012  -> PASS (bias demonstrated)  (bias=+0.0497)
  [7 Missing] MNAR: MMRM (biased too — MAR-only tool): nominal=-0.08  empirical=-0.0351  MCSE=0.0012  -> expected bias shown  (bias=+0.0449)
  [7 Missing] MNAR: sample lower bound centres on population lower: nominal=-0.0993  empirical=-0.1005  MCSE=0.0017  -> PASS
  [7 Missing] MNAR: sample upper bound centres on population upper: nominal=0.0442  empirical=0.042  MCSE=0.0016  -> PASS
  [7 Missing] MNAR: per-sample bracket rate (descriptive): nominal=~0.5 when truth sits at a bound edge  empirical=0.68  MCSE=0.019  -> note: geometry, see markdown  (truth -0.080 vs pop bounds [-0.099, +0.044])
[C7 MNAR done in 524s] failures=0


## Check 8 — Edge cases: the pipeline must fail loudly, not lie quietly
Single-shot probes (not Monte Carlo): zero-event arm, all-tied durations, 30% attrition,
near-total compliance (Wald CACE ≈ ITT, v3's inline formula copied verbatim), and a
degenerate zero-variance covariate.

In [17]:
rng = np.random.default_rng(SEED + 14)
def probe(name, fn, expect):
    try:
        out = fn()
        ok = expect == "runs" and out
        record("8 Edge", name, expect, "ran, finite output" if ok else str(out)[:60],
               "-", "PASS" if ok else "FAIL")
    except ValueError as e:
        ok = expect == "loud ValueError"
        record("8 Edge", name, expect, f"ValueError: {str(e)[:70]}...", "-", "PASS" if ok else "FAIL")
    except Exception as e:
        record("8 Edge", name, expect, f"{type(e).__name__}: {str(e)[:60]}", "-",
               "PASS" if expect == "loud error" else "FAIL (wrong failure mode)")

# (i) zero-event arm -> informative refusal from fit_cox_st
s0 = sim_cox_trial(rng, sigma_b=0.0); s0.loc[s0.arm == "unrestricted", "event"] = 0
probe("zero-event arm", lambda: sap.fit_cox_st(s0), "loud ValueError")

# (ii) heavy ties (durations rounded to 3 values) -> Efron handles, finite estimate
s1 = sim_cox_trial(rng, sigma_b=0.0); s1["duration_s"] = s1["duration_s"].clip(1, 45).round(-1).clip(lower=10)
probe("all-tied durations (Efron)", lambda: np.isfinite(sap.cox_lincom(sap.fit_cox_st(s1), {"unrestricted": 1})[0]), "runs")

# (iii) 30% attrition -> gatekeeper still fits; immediate contrast unaffected
a3 = sim_acc_trial(rng, b_unr=-0.10, drop_delayed=0.30)
probe("30% attrition gatekeeper", lambda: np.isfinite(sap.gatekeeper(a3, set(a3.pid), ALPHA, DELTA)["b2"]), "runs")

# (iv) near-total compliance: Wald CACE ~ ITT. Both arms engage heavily with their OWN
# tool (google ~0.97 with search, unrestricted ~0.97 with the LLM). The pre-fix v3
# formula divided by the CROSS-ARM engagement difference (~0), exploding the ratio; the
# corrected one-sided-noncompliance formula instruments assigned-LLM use, which is
# structurally zero in the Google arm, so first stage = treated engagement (~0.97).
acc_pw = sim_acc_trial(rng, b_unr=-0.10)
pooled = acc_pw.groupby("pid").agg(acc=("acc","mean"), arm=("arm","first")).reset_index()
eng = pd.Series(1, index=pooled.index); eng.iloc[rng.choice(len(pooled), 8, replace=False)] = 0
d = pooled.assign(engaged=eng.values); d = d[d.arm.isin(["google","unrestricted"])]
z = (d.arm == "unrestricted").astype(int)
itt = d.acc[z==1].mean() - d.acc[z==0].mean()
first_old = d.engaged[z==1].mean() - d.engaged[z==0].mean()          # pre-fix cross-arm
cace_old = itt/first_old if abs(first_old) > 1e-9 else np.nan
first_new = d.engaged[z==1].mean()                                    # corrected one-sided
cace_new = itt/first_new if first_new > 1e-9 else np.nan
record("8 Edge", "near-total compliance: corrected one-sided CACE ~ ITT", "CACE~ITT",
       f"ITT={itt:+.4f}, first={first_new:+.3f}, CACE={cace_new:+.4f}", "-",
       "PASS" if abs(cace_new-itt) < 0.02 else "FAIL")
record("8 Edge", "pre-fix cross-arm Wald (defect demonstration)", "explodes/undefined",
       f"first={first_old:+.3f}, CACE={'undefined' if np.isnan(cace_old) else f'{cace_old:+.3f}'}", "-",
       "PASS (defect demonstrated)" if (np.isnan(cace_old) or abs(cace_old) > 3*abs(itt) + 1e-9) else "note: benign here")
DISCREPANCIES.append(
    "PIPELINE DEFECT FOUND AND FIXED DURING VALIDATION: v3's CACE used the cross-arm engagement "
    "difference as the Wald first stage; with both arms engaging their own tools that denominator "
    "is ~0 and the ratio explodes. Fixed to the one-sided-noncompliance formulation (exposure = "
    "assigned-LLM use, structurally 0 in the Google arm; first stage = treated engagement). "
    "ACTION: the SAP's compliance paragraph should name this exposure definition explicitly.")

# (v) degenerate covariate -> informative refusal from the covariate guard
a4 = sim_acc_trial(rng); a4["aiuse_c"] = 0.0
probe("zero-variance covariate", lambda: sap.fit_primary_acc(a4), "loud ValueError")
print("edge probes complete")

  [8 Edge] zero-event arm: nominal=loud ValueError  empirical=ValueError: [fit_cox_st] at least one arm has ZERO events (correct answers): {'0.0...  MCSE=-  -> PASS


  [8 Edge] all-tied durations (Efron): nominal=runs  empirical=ran, finite output  MCSE=-  -> PASS


  [8 Edge] 30% attrition gatekeeper: nominal=runs  empirical=ran, finite output  MCSE=-  -> PASS
  [8 Edge] near-total compliance: corrected one-sided CACE ~ ITT: nominal=CACE~ITT  empirical=ITT=-0.1251, first=+0.955, CACE=-0.1311  MCSE=-  -> PASS
  [8 Edge] pre-fix cross-arm Wald (defect demonstration): nominal=explodes/undefined  empirical=first=+0.000, CACE=undefined  MCSE=-  -> PASS (defect demonstrated)
  [8 Edge] zero-variance covariate: nominal=loud ValueError  empirical=ValueError: [fit_primary_acc] model covariate 'aiuse_c' has ~zero variance (var=0....  MCSE=-  -> PASS
edge probes complete


## Summary — check × DGP × nominal × empirical × MC SE × verdict

In [18]:
summary = pd.DataFrame(RESULTS)[["check","dgp","nominal","empirical","mc_se","verdict","note"]]
pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 72); pd.set_option("display.width", 200)
print(summary.to_string(index=False))
vc = summary["verdict"].str.split(" ").str[0].value_counts()
print("\nVerdict counts:", vc.to_dict())
print("\nRuntime by check (s):", {k: round(v) for k, v in TIMES.items()})

    check                                                               dgp                              nominal                                                                             empirical    mc_se                             verdict                                                                                                        note
 1 Type-I                                   global null: H1a one-sided size                                 0.05                                                                                0.0408   0.0057                                PASS                                                                                                            
 1 Type-I                                 global null: FWER any false claim                               <=0.05                                                                                0.0408   0.0057                                PASS                                                     

## Discrepancy report — for the author to decide (nothing in the thesis text was edited)

In [19]:
if DISCREPANCIES:
    for i, d in enumerate(DISCREPANCIES, 1):
        print(f"[D{i}] {d}\n")
else:
    print("No discrepancies beyond MC error were detected.")
print("Reminder (pre-existing, found during code reading, quantified in Check 3): the NI-power alpha")
print("inconsistency also sits in appendix_power.tex (l.129 says 90% CI; l.144 caption says alpha=.025).")

[D1] PIPELINE DEFECT FOUND AND FIXED DURING VALIDATION: the original fit_primary_acc used a bare MixedLM.fit(method='lbfgs'); on realistic 16-item binomial-proportion outcomes this raised LinAlgError('Singular matrix') whenever the profiled random-intercept variance neared zero (100% of smoke-test draws). sap_estimators.py now carries a pre-specified optimizer fallback chain (lbfgs -> bfgs -> powell -> cg), identical wherever lbfgs succeeds. ACTION: add one line to the SAP software paragraph naming this chain before registration.

[D2] H4a power at extra=5pp: script t-test 0.283 vs pipeline GEE 0.160 — appendix_power.tex quotes the t-test numbers but the SAP's test is the GEE/GLMM.

[D3] H4a power at extra=10pp: script t-test 0.668 vs pipeline GEE 0.497 — appendix_power.tex quotes the t-test numbers but the SAP's test is the GEE/GLMM.

[D4] H4a power at extra=15pp: script t-test 0.967 vs pipeline GEE 0.843 — appendix_power.tex quotes the t-test numbers but the SAP's test is the GEE/GLM